# Notebook 9: Machine Learning Evaluation and Comparison

## Objective

This notebook conducts the final out-of-sample evaluation of the forecasting models developed in Notebook 08.

The analysis uses the locked January–December 2025 test period to:

1. evaluate the persistence, Ridge, Random Forest, and XGBoost forecasts
2. compare symmetric and asymmetric exchange-rate representations
3. examine performance across food subclasses and months
4. identify where forecasting errors are concentrate
5. compare machine-learning forecasts with time-aligned econometric baselines

No models are tuned or selected using the test results.

In [17]:
# import libraries
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

sns.set_theme(style="whitegrid")

print("Evaluation libraries imported successfully.")

Evaluation libraries imported successfully.


In [18]:
# define input and output locations
processed_data_directory = Path("../data/processed")
ml_table_directory = Path("../reports/tables/machine_learning")
econometric_table_directory = Path(
    "../reports/tables/econometrics"
)

evaluation_table_directory = Path(
    "../reports/tables/model_evaluation"
)
evaluation_figure_directory = Path(
    "../reports/figures/model_evaluation"
)

evaluation_table_directory.mkdir(parents=True, exist_ok=True)
evaluation_figure_directory.mkdir(parents=True, exist_ok=True)


# load the locked outcomes and modelling outputs
ml_data = pd.read_csv(
    processed_data_directory / "ml_model_data.csv",
    parse_dates=["Date"],
)

ml_test_predictions = pd.read_csv(
    ml_table_directory / "ml_test_predictions.csv",
    parse_dates=["Date"],
)

validation_model_results = pd.read_csv(
    ml_table_directory / "validation_model_comparison.csv"
)

test_actuals = ml_data.loc[
    ml_data["Split"] == "Test",
    [
        "Date",
        "ClassDescription",
        "SubclassDescription",
        "Food_Inflation_Pct",
    ],
].copy()

print("Machine-learning data loaded:", len(ml_data))
print("Locked test outcomes loaded:", len(test_actuals))
print("Forecast rows loaded:", len(ml_test_predictions))
print("Validation models loaded:", len(validation_model_results))

Machine-learning data loaded: 4554
Locked test outcomes loaded: 552
Forecast rows loaded: 3864
Validation models loaded: 7


In [19]:
# validate the evaluation sample
identifier_columns = [
    "Date",
    "ClassDescription",
    "SubclassDescription",
]

prediction_identifier_columns = identifier_columns + [
    "Model",
    "Representation",
]

prediction_counts = (
    ml_test_predictions.groupby(
        ["Model", "Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Predictions"})
)

evaluation_checks = pd.DataFrame(
    {
        "Check": [
            "Test sample contains 552 outcomes",
            "Test sample covers 46 food subclasses",
            "Test sample covers 12 months",
            "Test outcomes contain no duplicate keys",
            "Predictions contain seven model variants",
            "Every model variant contains 552 predictions",
            "Predictions contain no duplicate records",
            "Target was absent from prediction file",
        ],
        "Passed": [
            len(test_actuals) == 552,
            test_actuals["SubclassDescription"].nunique() == 46,
            test_actuals["Date"].nunique() == 12,
            not test_actuals.duplicated(identifier_columns).any(),
            len(prediction_counts) == 7,
            prediction_counts["Predictions"].eq(552).all(),
            not ml_test_predictions.duplicated(
                prediction_identifier_columns
            ).any(),
            "Food_Inflation_Pct"
            not in ml_test_predictions.columns,
        ],
    }
)

if not evaluation_checks["Passed"].all():
    failed_checks = evaluation_checks.loc[
        ~evaluation_checks["Passed"],
        "Check",
    ].tolist()
    raise ValueError(f"Evaluation checks failed: {failed_checks}")


# Attach actual outcomes for final evaluation
forecast_evaluation_data = ml_test_predictions.merge(
    test_actuals,
    on=identifier_columns,
    how="left",
    validate="many_to_one",
)

missing_actuals = int(
    forecast_evaluation_data["Food_Inflation_Pct"]
    .isna()
    .sum()
)

if missing_actuals:
    raise ValueError(
        f"{missing_actuals} predictions could not be matched "
        "to test outcomes."
    )

evaluation_sample_summary = pd.DataFrame(
    {
        "Value": [
            len(test_actuals),
            test_actuals["SubclassDescription"].nunique(),
            test_actuals["Date"].nunique(),
            test_actuals["Date"].min(),
            test_actuals["Date"].max(),
            len(prediction_counts),
            len(forecast_evaluation_data),
            missing_actuals,
        ]
    },
    index=[
        "Test outcomes",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Forecast variants",
        "Forecast-evaluation rows",
        "Missing matched outcomes",
    ],
)

display(evaluation_checks)
display(evaluation_sample_summary)
display(prediction_counts)

print(
    "All evaluation setup checks passed:",
    bool(evaluation_checks["Passed"].all()),
)

,Check,Passed
0,Test sample contains 552 outcomes,True
1,Test sample covers 46 food subclasses,True
2,Test sample covers 12 months,True
3,Test outcomes contain no duplicate keys,True
4,Predictions contain seven model variants,True
5,Every model variant contains 552 predictions,True
6,Predictions contain no duplicate records,True
7,Target was absent from prediction file,True


,Value
Test outcomes,552
Food subclasses,46
Unique months,12
Start date,2025-01-01 00:00:00
End date,2025-12-01 00:00:00
Forecast variants,7
Forecast-evaluation rows,3864
Missing matched outcomes,0


,Model,Representation,Predictions
0,Persistence,Lag-1 benchmark,552
1,Random Forest,Asymmetric,552
2,Random Forest,Symmetric,552
3,Ridge,Asymmetric,552
4,Ridge,Symmetric,552
5,XGBoost,Asymmetric,552
6,XGBoost,Symmetric,552


All evaluation setup checks passed: True


## Overall locked-test performance

The following evaluation compares all seven forecast variants across the complete 2025 test sample.

In addition to MAE, RMSE, R², and directional accuracy, mean bias is reported. A bias is calculated as predicted inflation minus observed inflation. A positive value therefore indicates average overprediction, while a negative value indicates average underprediction.

In [20]:
# calculate overall test metrics
def calculate_forecast_metrics(evaluation_data):
    actual_values = evaluation_data[
        "Food_Inflation_Pct"
    ].to_numpy()

    predicted_values = evaluation_data[
        "Predicted_Food_Inflation_Pct"
    ].to_numpy()

    forecast_errors = predicted_values - actual_values

    return {
        "Observations": len(evaluation_data),
        "MAE": mean_absolute_error(
            actual_values,
            predicted_values,
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                actual_values,
                predicted_values,
            )
        ),
        "R2": r2_score(
            actual_values,
            predicted_values,
        ),
        "Directional_Accuracy_Pct": (
            np.mean(
                np.sign(actual_values)
                == np.sign(predicted_values)
            )
            * 100
        ),
        "Mean_Bias": np.mean(forecast_errors),
    }


test_metric_records = []

for (
    model_name,
    representation,
), model_data in forecast_evaluation_data.groupby(
    ["Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(model_data)

    test_metric_records.append(
        {
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

ml_test_model_results = pd.DataFrame(test_metric_records)

persistence_result = ml_test_model_results.loc[
    ml_test_model_results["Model"] == "Persistence"
].iloc[0]

ml_test_model_results[
    "RMSE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["RMSE"]
        - ml_test_model_results["RMSE"]
    )
    / persistence_result["RMSE"]
    * 100
)

ml_test_model_results[
    "MAE_Improvement_vs_Persistence_Pct"
] = (
    (
        persistence_result["MAE"]
        - ml_test_model_results["MAE"]
    )
    / persistence_result["MAE"]
    * 100
)

ml_test_model_results["Test_Rank"] = (
    ml_test_model_results["RMSE"]
    .rank(method="min")
    .astype(int)
)

ml_test_model_results = ml_test_model_results.sort_values(
    ["Test_Rank", "MAE"]
).reset_index(drop=True)

display(
    ml_test_model_results[
        [
            "Test_Rank",
            "Model",
            "Representation",
            "Observations",
            "MAE",
            "RMSE",
            "R2",
            "Directional_Accuracy_Pct",
            "Mean_Bias",
            "RMSE_Improvement_vs_Persistence_Pct",
            "MAE_Improvement_vs_Persistence_Pct",
        ]
    ]
)

leading_test_model = ml_test_model_results.iloc[0]

print("Lowest test RMSE:")
print(
    f"{leading_test_model['Model']} — "
    f"{leading_test_model['Representation']}"
)
print(f"Test RMSE: {leading_test_model['RMSE']:.6f}")

,Test_Rank,Model,Representation,Observations,MAE,RMSE,R2,Directional_Accuracy_Pct,Mean_Bias,RMSE_Improvement_vs_Persistence_Pct,MAE_Improvement_vs_Persistence_Pct
0,1,XGBoost,Symmetric,552,1.060854,1.929075,0.113069,59.782609,0.263793,13.703894,20.423091
1,2,Random Forest,Symmetric,552,1.066689,1.964122,0.080550,63.405797,0.314879,12.136122,19.985359
2,3,XGBoost,Asymmetric,552,1.090656,1.973192,0.072038,61.231884,0.323877,11.730351,18.187517
3,4,Ridge,Symmetric,552,1.089515,1.984985,0.060913,60.869565,0.280788,11.202815,18.273135
4,5,Ridge,Asymmetric,552,1.092858,1.986590,0.059394,61.050725,0.287840,11.131018,18.022346
5,6,Random Forest,Asymmetric,552,1.075816,1.991030,0.055185,61.231884,0.313877,10.932393,19.300687
6,7,Persistence,Lag-1 benchmark,552,1.333117,2.235414,-0.190988,54.347826,0.025886,0.000000,0.000000


Lowest test RMSE:
XGBoost — Symmetric
Test RMSE: 1.929075


In [21]:
# examine the validation-to-test generalisation gap
validation_metrics = validation_model_results.rename(
    columns={
        "MAE": "Validation_MAE",
        "RMSE": "Validation_RMSE",
        "R2": "Validation_R2",
        "Directional_Accuracy_Pct": (
            "Validation_Directional_Accuracy_Pct"
        ),
    }
)

test_metrics = ml_test_model_results.rename(
    columns={
        "MAE": "Test_MAE",
        "RMSE": "Test_RMSE",
        "R2": "Test_R2",
        "Directional_Accuracy_Pct": (
            "Test_Directional_Accuracy_Pct"
        ),
    }
)

validation_test_comparison = validation_metrics.merge(
    test_metrics[
        [
            "Model",
            "Representation",
            "Test_MAE",
            "Test_RMSE",
            "Test_R2",
            "Test_Directional_Accuracy_Pct",
            "Mean_Bias",
            "Test_Rank",
        ]
    ],
    on=["Model", "Representation"],
    how="inner",
    validate="one_to_one",
)

validation_test_comparison[
    "RMSE_Generalisation_Gap"
] = (
    validation_test_comparison["Test_RMSE"]
    - validation_test_comparison["Validation_RMSE"]
)

validation_test_comparison[
    "RMSE_Change_Pct"
] = (
    validation_test_comparison["RMSE_Generalisation_Gap"]
    / validation_test_comparison["Validation_RMSE"]
    * 100
)

validation_test_comparison["Validation_Rank"] = (
    validation_test_comparison["Validation_RMSE"]
    .rank(method="min")
    .astype(int)
)

validation_test_comparison = validation_test_comparison.sort_values(
    "Test_Rank"
).reset_index(drop=True)

display(
    validation_test_comparison[
        [
            "Model",
            "Representation",
            "Validation_Rank",
            "Test_Rank",
            "Validation_RMSE",
            "Test_RMSE",
            "RMSE_Generalisation_Gap",
            "RMSE_Change_Pct",
            "Validation_R2",
            "Test_R2",
            "Validation_Directional_Accuracy_Pct",
            "Test_Directional_Accuracy_Pct",
        ]
    ]
)

,Model,Representation,Validation_Rank,Test_Rank,Validation_RMSE,Test_RMSE,RMSE_Generalisation_Gap,RMSE_Change_Pct,Validation_R2,Test_R2,Validation_Directional_Accuracy_Pct,Test_Directional_Accuracy_Pct
0,XGBoost,Symmetric,1,1,1.722070,1.929075,0.207006,12.020751,0.177449,0.113069,63.949275,59.782609
1,Random Forest,Symmetric,3,2,1.759446,1.964122,0.204676,11.632964,0.141356,0.080550,64.855072,63.405797
2,XGBoost,Asymmetric,2,3,1.728267,1.973192,0.244925,14.171711,0.171518,0.072038,63.405797,61.231884
3,Ridge,Symmetric,5,4,1.869568,1.984985,0.115417,6.173462,0.030509,0.060913,63.768116,60.869565
4,Ridge,Asymmetric,6,5,1.873179,1.986590,0.113410,6.054428,0.026759,0.059394,63.586957,61.050725
5,Random Forest,Asymmetric,4,6,1.796596,1.991030,0.194434,10.822330,0.104713,0.055185,63.949275,61.231884
6,Persistence,Lag-1 benchmark,7,7,2.340242,2.235414,-0.104828,-4.479351,-0.519088,-0.190988,56.340580,54.347826


### Interpretation of overall test performance

The validation-based selection was supported by the locked-test results. Symmetric XGBoost retained first place, with an RMSE of 1.929 and MAE of 1.061. It reduced RMSE by 13.7% and MAE by 20.4% relative to persistence.

All six machine-learning models outperformed persistence and produced positive test R² values. Persistence produced a negative R², indicating that it performed worse than predicting the test-sample mean.

Symmetric Random Forest ranked second by RMSE and achieved the highest
directional accuracy of 63.41%. Therefore, the model that minimised the size of forecast errors was not the model that most frequently predicted the correct sign of food inflation.

Test errors were higher than validation errors for all machine-learning models. For symmetric XGBoost, RMSE increased by approximately 12.0%, while R² declined from 0.177 to 0.113. This deterioration demonstrates the importance of retaining a genuinely unseen test period.

All machine-learning models had positive mean bias, indicating modest average overprediction during 2025. Nevertheless, they remained more accurate than the persistence benchmark.

The symmetric specification achieved a lower overall test RMSE than its
asymmetric counterpart for Ridge, Random Forest, and XGBoost. Subclass-level analysis is required to determine whether asymmetric features nevertheless benefited particular food categories.

In [22]:
# calculate metrics for every model-subclass combination
subclass_metric_records = []

grouping_columns = [
    "ClassDescription",
    "SubclassDescription",
    "Model",
    "Representation",
]

for group_values, group_data in forecast_evaluation_data.groupby(
    grouping_columns,
    sort=False,
):
    (
        class_description,
        subclass_description,
        model_name,
        representation,
    ) = group_values

    metrics = calculate_forecast_metrics(group_data)

    subclass_metric_records.append(
        {
            "ClassDescription": class_description,
            "SubclassDescription": subclass_description,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

subclass_model_results = pd.DataFrame(subclass_metric_records)

print(
    "Subclass-model results:",
    len(subclass_model_results),
)
print(
    "Expected results:",
    46 * 7,
)
print(
    "Observations per result:",
    sorted(
        subclass_model_results["Observations"].unique()
    ),
)

Subclass-model results: 322
Expected results: 322
Observations per result: [np.int64(12)]


In [23]:
# identify the lowest-RMSE model for each food subclass
subclass_winners = (
    subclass_model_results.sort_values(
        [
            "SubclassDescription",
            "RMSE",
            "MAE",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription"],
        keep="first",
    )
    .rename(
        columns={
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    [
        [
            "ClassDescription",
            "SubclassDescription",
            "Winning_Model",
            "Winning_Representation",
            "Winning_MAE",
            "Winning_RMSE",
            "Winning_R2",
            "Winning_Directional_Accuracy_Pct",
        ]
    ]
    .reset_index(drop=True)
)

subclass_winner_summary = (
    subclass_winners.groupby(
        [
            "Winning_Model",
            "Winning_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        "Food_Subclasses",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(subclass_winner_summary)
display(
    subclass_winners.sort_values(
        "Winning_RMSE"
    ).head(10)
)

,Winning_Model,Winning_Representation,Food_Subclasses
0,Random Forest,Symmetric,16
1,XGBoost,Symmetric,7
2,Persistence,Lag-1 benchmark,5
3,Random Forest,Asymmetric,5
4,XGBoost,Asymmetric,5
5,Ridge,Symmetric,4
6,Ridge,Asymmetric,4


,ClassDescription,SubclassDescription,Winning_Model,Winning_Representation,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct
35,Soft drinks,Soft drinks,Persistence,Lag-1 benchmark,0.241626,0.294418,-0.250721,50.000000
0,Other food,Baby food,Persistence,Lag-1 benchmark,0.286480,0.339117,-1.241370,41.666667
12,Fruits and nuts,Fruit and nuts ground and other preparations,XGBoost,Symmetric,0.309381,0.372183,0.355309,75.000000
9,Fish and other seafood,Fish,Random Forest,Symmetric,0.314600,0.383227,-0.051256,75.000000
26,Other food,Other food products n.e.c.,Random Forest,Asymmetric,0.281066,0.408675,-0.220225,75.000000
1,Cereal products,Bread and bakery products,Ridge,Symmetric,0.315104,0.409537,-0.424188,66.666667
42,Vegetables,Vegetables and pulses ground and other prepara...,XGBoost,Symmetric,0.368402,0.434867,-0.154845,58.333333
17,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",XGBoost,Asymmetric,0.348134,0.460431,-0.516696,75.000000
5,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",XGBoost,Asymmetric,0.394256,0.498729,0.015257,83.333333
44,Water,Water,XGBoost,Symmetric,0.432528,0.524496,-0.036607,66.666667


In [24]:
# compare symmetric and asymmetric RMSE by subclass
ml_subclass_results = subclass_model_results.loc[
    subclass_model_results["Model"] != "Persistence"
].copy()

subclass_representation_comparison = (
    ml_subclass_results.pivot(
        index=[
            "ClassDescription",
            "SubclassDescription",
            "Model",
        ],
        columns="Representation",
        values="RMSE",
    )
    .reset_index()
)

subclass_representation_comparison.columns.name = None

subclass_representation_comparison[
    "Asymmetric_Improvement_Pct"
] = (
    (
        subclass_representation_comparison["Symmetric"]
        - subclass_representation_comparison["Asymmetric"]
    )
    / subclass_representation_comparison["Symmetric"]
    * 100
)

subclass_representation_comparison[
    "Preferred_Representation"
] = np.where(
    subclass_representation_comparison["Asymmetric"]
    < subclass_representation_comparison["Symmetric"],
    "Asymmetric",
    "Symmetric",
)

subclass_representation_summary = (
    subclass_representation_comparison.groupby(
        [
            "Model",
            "Preferred_Representation",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        ["Model", "Preferred_Representation"]
    )
    .reset_index(drop=True)
)

display(subclass_representation_summary)

print("Largest asymmetric improvements:")
display(
    subclass_representation_comparison.sort_values(
        "Asymmetric_Improvement_Pct",
        ascending=False,
    ).head(10)
)

,Model,Preferred_Representation,Food_Subclasses
0,Random Forest,Asymmetric,23
1,Random Forest,Symmetric,23
2,Ridge,Asymmetric,11
3,Ridge,Symmetric,35
4,XGBoost,Asymmetric,16
5,XGBoost,Symmetric,30


Largest asymmetric improvements:


,ClassDescription,SubclassDescription,Model,Asymmetric,Symmetric,Asymmetric_Improvement_Pct,Preferred_Representation
96,"Sugar, confectionery and desserts","Chocolate, cocoa, and cocoa-based food products",Random Forest,0.505572,0.575587,12.164121,Asymmetric
129,Vegetables,Vegetables and pulses ground and other prepara...,Random Forest,0.464160,0.525447,11.663939,Asymmetric
12,Cereal products,"Macaroni, noodles, couscous and similar pasta ...",Random Forest,0.498739,0.562427,11.323772,Asymmetric
86,Other food,"Salt, condiments and sauces",XGBoost,0.662000,0.742925,10.892848,Asymmetric
93,Soft drinks,Soft drinks,Random Forest,0.400876,0.445238,9.963738,Asymmetric
65,"Milk, other dairy products and eggs",Other milk and cream,XGBoost,0.739107,0.812784,9.064772,Asymmetric
87,Other food,"Spices, culinary herbs and seeds",Random Forest,0.927382,1.019095,8.999465,Asymmetric
71,Oils and fats,Margarine and similar preparations,XGBoost,0.789302,0.844154,6.497936,Asymmetric
113,Tea,Tea and other plant products for infusion,XGBoost,0.769550,0.816799,5.784706,Asymmetric
63,"Milk, other dairy products and eggs",Other milk and cream,Random Forest,0.785311,0.829602,5.338785,Asymmetric


### Interpretation of subclass-level performance

Machine-learning models achieved the lowest subclass RMSE in 41 of the 46 food subclasses. Persistence was the strongest forecast in only five subclasses, showing that machine learning generally added predictive value beyond the previous month's inflation rate.

Symmetric Random Forest was the most frequent subclass winner, ranking first in 16 categories. Symmetric XGBoost won seven categories, while the remaining categories were distributed across asymmetric tree models, Ridge models, and persistence. Thus, the best pooled model was not automatically the best model for every food category.

Symmetric specifications were the winning forecasts for 27 subclasses,
asymmetric specifications for 14, and persistence for five. Within each
algorithm, symmetric features were preferred for 35 of 46 Ridge comparisons and 30 of 46 XGBoost comparisons. Random Forest was evenly divided, with each representation preferred for 23 subclasses.

Across all three algorithms, asymmetric features reduced subclass RMSE in 50 of 138 comparisons. Their forecasting value was therefore category-specific rather than universal. The largest asymmetric improvements were approximately 12.2% for chocolate and cocoa products under Random Forest, 11.7% for prepared vegetable and pulse products, and 11.3% for pasta products.

These subclass winners are retrospective test-period findings and are not used to refit or select new models. Furthermore, each subclass result is based on only 12 observations, so individual rankings and R² values should be interpreted cautiously.

In [25]:
# calculate metrics for each month and model
monthly_metric_records = []

for (
    forecast_month,
    model_name,
    representation,
), monthly_data in forecast_evaluation_data.groupby(
    ["Date", "Model", "Representation"],
    sort=False,
):
    metrics = calculate_forecast_metrics(monthly_data)

    monthly_metric_records.append(
        {
            "Date": forecast_month,
            "Model": model_name,
            "Representation": representation,
            **metrics,
        }
    )

monthly_model_results = pd.DataFrame(monthly_metric_records)

print("Monthly model results:", len(monthly_model_results))
print("Expected results:", 12 * 7)
print(
    "Observations per monthly result:",
    sorted(monthly_model_results["Observations"].unique()),
)

Monthly model results: 84
Expected results: 84
Observations per monthly result: [np.int64(46)]


In [26]:
# identify the lowest-RMSE forecast in each month
monthly_winners = (
    monthly_model_results.sort_values(
        ["Date", "RMSE", "MAE"]
    )
    .drop_duplicates(subset=["Date"], keep="first")
    .rename(
        columns={
            "Model": "Winning_Model",
            "Representation": "Winning_Representation",
            "MAE": "Winning_MAE",
            "RMSE": "Winning_RMSE",
            "R2": "Winning_R2",
            "Directional_Accuracy_Pct": (
                "Winning_Directional_Accuracy_Pct"
            ),
        }
    )
    [
        [
            "Date",
            "Winning_Model",
            "Winning_Representation",
            "Winning_MAE",
            "Winning_RMSE",
            "Winning_R2",
            "Winning_Directional_Accuracy_Pct",
        ]
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

monthly_winner_summary = (
    monthly_winners.groupby(
        ["Winning_Model", "Winning_Representation"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Months_Won"})
    .sort_values("Months_Won", ascending=False)
    .reset_index(drop=True)
)

display(monthly_winners)
display(monthly_winner_summary)

,Date,Winning_Model,Winning_Representation,Winning_MAE,Winning_RMSE,Winning_R2,Winning_Directional_Accuracy_Pct
0,2025-01-01,Random Forest,Symmetric,1.199331,1.932682,0.035610,54.347826
1,2025-02-01,XGBoost,Symmetric,0.864124,1.194605,0.411526,71.739130
2,2025-03-01,Random Forest,Asymmetric,1.015759,1.752533,0.032738,67.391304
3,2025-04-01,Random Forest,Symmetric,1.079214,2.032790,0.115870,65.217391
4,2025-05-01,Persistence,Lag-1 benchmark,1.146359,1.640086,0.542885,47.826087
5,2025-06-01,Random Forest,Asymmetric,0.927133,1.481960,0.178227,67.391304
6,2025-07-01,Persistence,Lag-1 benchmark,1.244300,1.713075,0.502467,56.521739
7,2025-08-01,XGBoost,Symmetric,1.006774,1.554883,0.105164,45.652174
8,2025-09-01,Persistence,Lag-1 benchmark,0.961552,1.566576,0.526864,71.739130
9,2025-10-01,Persistence,Lag-1 benchmark,1.108263,1.764813,0.398771,54.347826


,Winning_Model,Winning_Representation,Months_Won
0,Persistence,Lag-1 benchmark,4
1,Random Forest,Symmetric,4
2,Random Forest,Asymmetric,2
3,XGBoost,Symmetric,2


In [27]:
# describe monthly inflation dispersion and forecast difficulty
monthly_actual_summary = (
    test_actuals.groupby("Date", as_index=False)
    .agg(
        Mean_Actual_Inflation=(
            "Food_Inflation_Pct",
            "mean",
        ),
        Actual_Inflation_Std=(
            "Food_Inflation_Pct",
            "std",
        ),
        Minimum_Actual_Inflation=(
            "Food_Inflation_Pct",
            "min",
        ),
        Maximum_Actual_Inflation=(
            "Food_Inflation_Pct",
            "max",
        ),
    )
)

monthly_difficulty = monthly_actual_summary.merge(
    monthly_winners[
        [
            "Date",
            "Winning_Model",
            "Winning_Representation",
            "Winning_RMSE",
        ]
    ],
    on="Date",
    how="left",
    validate="one_to_one",
)

monthly_difficulty["Date"] = (
    monthly_difficulty["Date"].dt.strftime("%Y-%m")
)

display(
    monthly_difficulty.sort_values(
        "Winning_RMSE",
        ascending=False,
    )
)

,Date,Mean_Actual_Inflation,Actual_Inflation_Std,Minimum_Actual_Inflation,Maximum_Actual_Inflation,Winning_Model,Winning_Representation,Winning_RMSE
10,2025-11,0.301289,2.346504,-8.111348,12.188982,Random Forest,Symmetric,2.160644
3,2025-04,0.596938,2.185783,-4.299728,11.442515,Random Forest,Symmetric,2.032790
0,2025-01,-0.084558,1.989788,-8.773891,5.164323,Random Forest,Symmetric,1.932682
9,2025-10,-0.042564,2.301185,-10.178411,5.070213,Persistence,Lag-1 benchmark,1.764813
2,2025-03,0.316300,1.801635,-5.654528,6.960291,Random Forest,Asymmetric,1.752533
6,2025-07,-0.161781,2.455489,-12.438372,6.057494,Persistence,Lag-1 benchmark,1.713075
4,2025-05,0.691416,2.452599,-3.005558,14.584680,Persistence,Lag-1 benchmark,1.640086
8,2025-09,-0.420810,2.302667,-12.977642,1.429981,Persistence,Lag-1 benchmark,1.566576
7,2025-08,0.102587,1.661876,-6.997019,2.317484,XGBoost,Symmetric,1.554883
5,2025-06,0.116058,1.652849,-6.891490,3.447723,Random Forest,Asymmetric,1.481960


### Interpretation of monthly performance

Machine-learning models achieved the lowest RMSE in eight of the twelve test months, while persistence was preferred in May, July, September, and October. This indicates that recent observed inflation sometimes provided a strong short-term benchmark.

Random Forest was the most frequent monthly winner. Its symmetric representation won four months and its asymmetric representation won two. Symmetric XGBoost won February and August.

Although symmetric XGBoost won only two individual months, it achieved the lowest RMSE across the complete test year. Overall RMSE depends on the magnitude of every forecast error rather than the number of monthly wins. A model can therefore rank first overall by avoiding especially large errors without being the lowest-error model in most individual months.

November, April, and January were the most difficult forecast months based on the lowest attainable monthly RMSE. February and December were comparatively easier. Monthly performance therefore varied substantially, supporting the use of a full twelve-month evaluation instead of relying on isolated periods.

In [28]:
# inspect available econometric data and forecast outputs
econometric_data_path = (
    processed_data_directory / "econometric_model_data.csv"
)

econometric_data = pd.read_csv(
    econometric_data_path,
    parse_dates=["Date"],
)

data_column_inventory = pd.concat(
    [
        pd.DataFrame(
            {
                "Dataset": "Econometric",
                "Column": econometric_data.columns,
            }
        ),
        pd.DataFrame(
            {
                "Dataset": "Machine learning",
                "Column": ml_data.columns,
            }
        ),
    ],
    ignore_index=True,
)

reports_table_directory = Path("../reports/tables")
existing_csv_files = sorted(
    reports_table_directory.rglob("*.csv")
)

var_or_forecast_files = [
    file_path
    for file_path in existing_csv_files
    if (
        "var" in file_path.stem.lower()
        or "forecast" in file_path.stem.lower()
    )
]

econometric_input_summary = pd.DataFrame(
    {
        "Value": [
            len(econometric_data),
            econometric_data[
                "SubclassDescription"
            ].nunique(),
            econometric_data["Date"].nunique(),
            econometric_data["Date"].min(),
            econometric_data["Date"].max(),
            int(econometric_data.isna().sum().sum()),
        ]
    },
    index=[
        "Observations",
        "Food subclasses",
        "Unique months",
        "Start date",
        "End date",
        "Missing values",
    ],
)

display(econometric_input_summary)
display(data_column_inventory)

print("Existing VAR or forecast-related tables:")

if var_or_forecast_files:
    for file_path in var_or_forecast_files:
        print(file_path.as_posix())
else:
    print("None found.")

,Value
Observations,4830
Food subclasses,46
Unique months,105
Start date,2017-04-01 00:00:00
End date,2025-12-01 00:00:00
Missing values,0


,Dataset,Column
0,Econometric,Date
1,Econometric,GroupDescription
2,Econometric,ClassDescription
3,Econometric,SubclassDescription
4,Econometric,Subclass_Weight
5,Econometric,CPI
6,Econometric,Log_CPI
7,Econometric,Food_Inflation_Pct
8,Econometric,ExchangeRate
9,Econometric,Log_ExchangeRate


Existing VAR or forecast-related tables:
None found.


In [29]:
# inspect archived VAR forecast results
var_archive_directory = Path("../reports/archive/var")

var_actual_vs_forecast = pd.read_csv(
    var_archive_directory / "var_actual_vs_forecast.csv"
)

var_forecast_comparison = pd.read_csv(
    var_archive_directory / "var_forecast_comparison.csv"
)

var_file_summary = pd.DataFrame(
    [
        {
            "File": "var_actual_vs_forecast.csv",
            "Rows": len(var_actual_vs_forecast),
            "Columns": len(var_actual_vs_forecast.columns),
            "Missing_Values": int(
                var_actual_vs_forecast.isna().sum().sum()
            ),
        },
        {
            "File": "var_forecast_comparison.csv",
            "Rows": len(var_forecast_comparison),
            "Columns": len(var_forecast_comparison.columns),
            "Missing_Values": int(
                var_forecast_comparison.isna().sum().sum()
            ),
        },
    ]
)

var_column_inventory = pd.DataFrame(
    [
        {
            "File": "var_actual_vs_forecast.csv",
            "Column": column,
        }
        for column in var_actual_vs_forecast.columns
    ]
    + [
        {
            "File": "var_forecast_comparison.csv",
            "Column": column,
        }
        for column in var_forecast_comparison.columns
    ]
)

display(var_file_summary)
display(var_column_inventory)

print("VAR actual-versus-forecast preview:")
display(var_actual_vs_forecast.head(10))

print("VAR forecast-comparison table:")
display(var_forecast_comparison)

,File,Rows,Columns,Missing_Values
0,var_actual_vs_forecast.csv,12,2,0
1,var_forecast_comparison.csv,2830,2,0


,File,Column
0,var_actual_vs_forecast.csv,Actual CPI
1,var_actual_vs_forecast.csv,Forecast CPI
2,var_forecast_comparison.csv,Actual CPI
3,var_forecast_comparison.csv,Forecast CPI


VAR actual-versus-forecast preview:


,Actual CPI,Forecast CPI
0,101.000000,95.721322
1,100.600000,95.318089
2,100.900000,95.281598
3,100.600000,93.474067
4,101.800000,92.181029
5,101.700000,91.855426
6,102.700000,91.749538
7,103.400000,89.966545
8,103.400000,89.166436
9,102.700000,88.774040


VAR forecast-comparison table:


,Actual CPI,Forecast CPI
0,85.200000,83.709020
1,83.800000,82.487946
2,84.500000,82.742504
3,84.200000,84.119248
4,85.400000,83.739605
...,...,...
2825,103.400000,80.523885
2826,103.400000,80.523885
2827,102.700000,80.523885
2828,103.000000,80.523885


### Econometric forecast-benchmark design

No previously generated VAR or econometric forecast tables were found.
Consequently, forecast results cannot be inferred from the full-sample
cointegration models.

Separate short-run ARDL and NARDL forecast benchmarks are constructed for the comparison:

- the symmetric ARDL benchmark uses lagged food inflation and lagged symmetric exchange-rate changes
- the asymmetric NARDL benchmark uses lagged food inflation together with separate lagged depreciation and appreciation shocks
- monthly seasonal indicators are included
- candidate lag orders range from one to six
- all candidates use a common six-month hold-back
- BIC selection uses development data ending in December 2024
- predictions cover January–December 2025

Contemporaneous exchange-rate changes are excluded. This ensures that the econometric forecasts use lagged exchange-rate information consistent with the machine-learning forecasting exercise.

In [30]:
# construct lagged variables for econometric forecasting
maximum_forecast_lag = 6

econometric_forecast_data = (
    econometric_data.sort_values(
        ["SubclassDescription", "Date"]
    )
    .reset_index(drop=True)
    .copy()
)

lag_source_columns = [
    "Food_Inflation_Pct",
    "ExchangeRate_Log_Change_Pct",
    "Depreciation_Shock_Pct",
    "Appreciation_Shock_Pct",
]

for lag in range(1, maximum_forecast_lag + 1):
    for source_column in lag_source_columns:
        lagged_column = f"{source_column}_Lag{lag}"

        econometric_forecast_data[lagged_column] = (
            econometric_forecast_data.groupby(
                "SubclassDescription"
            )[source_column]
            .shift(lag)
        )


# Add monthly seasonal indicators
month_indicators = pd.get_dummies(
    econometric_forecast_data["Date"].dt.month,
    prefix="Month",
    drop_first=True,
    dtype=float,
)

econometric_forecast_data = pd.concat(
    [
        econometric_forecast_data,
        month_indicators,
    ],
    axis=1,
)

seasonal_columns = list(month_indicators.columns)

maximum_lag_columns = [
    f"{source_column}_Lag{maximum_forecast_lag}"
    for source_column in lag_source_columns
]

aligned_econometric_forecast_data = (
    econometric_forecast_data.dropna(
        subset=maximum_lag_columns
    )
    .copy()
)

forecast_development_data = (
    aligned_econometric_forecast_data.loc[
        aligned_econometric_forecast_data["Date"]
        <= pd.Timestamp("2024-12-01")
    ]
    .copy()
)

forecast_test_data = (
    aligned_econometric_forecast_data.loc[
        aligned_econometric_forecast_data["Date"].between(
            pd.Timestamp("2025-01-01"),
            pd.Timestamp("2025-12-01"),
        )
    ]
    .copy()
)

In [31]:
# validate the econometric forecasting sample
decomposition_error = np.abs(
    econometric_forecast_data[
        "ExchangeRate_Log_Change_Pct"
    ]
    - (
        econometric_forecast_data[
            "Depreciation_Shock_Pct"
        ]
        + econometric_forecast_data[
            "Appreciation_Shock_Pct"
        ]
    )
).max()

econometric_forecast_checks = pd.DataFrame(
    {
        "Check": [
            "Development sample contains 4,002 rows",
            "Development sample covers 87 months",
            "Development ends in December 2024",
            "Test sample contains 552 rows",
            "Test sample covers 12 months",
            "Test sample covers 46 subclasses",
            "No duplicate development rows",
            "No duplicate test rows",
            "No missing values in aligned data",
            "Depreciation shocks are non-negative",
            "Appreciation shocks are non-positive",
            "Shock decomposition is exact",
        ],
        "Passed": [
            len(forecast_development_data) == 4002,
            forecast_development_data["Date"].nunique() == 87,
            forecast_development_data["Date"].max()
            == pd.Timestamp("2024-12-01"),
            len(forecast_test_data) == 552,
            forecast_test_data["Date"].nunique() == 12,
            forecast_test_data[
                "SubclassDescription"
            ].nunique()
            == 46,
            not forecast_development_data.duplicated(
                identifier_columns
            ).any(),
            not forecast_test_data.duplicated(
                identifier_columns
            ).any(),
            not aligned_econometric_forecast_data.isna()
            .any()
            .any(),
            econometric_forecast_data[
                "Depreciation_Shock_Pct"
            ].min()
            >= 0,
            econometric_forecast_data[
                "Appreciation_Shock_Pct"
            ].max()
            <= 0,
            np.isclose(decomposition_error, 0.0),
        ],
    }
)

forecast_sample_summary = pd.DataFrame(
    [
        {
            "Sample": "Development",
            "Start_Date": forecast_development_data[
                "Date"
            ].min(),
            "End_Date": forecast_development_data[
                "Date"
            ].max(),
            "Observations": len(forecast_development_data),
            "Food_Subclasses": forecast_development_data[
                "SubclassDescription"
            ].nunique(),
            "Unique_Months": forecast_development_data[
                "Date"
            ].nunique(),
        },
        {
            "Sample": "Locked test",
            "Start_Date": forecast_test_data["Date"].min(),
            "End_Date": forecast_test_data["Date"].max(),
            "Observations": len(forecast_test_data),
            "Food_Subclasses": forecast_test_data[
                "SubclassDescription"
            ].nunique(),
            "Unique_Months": forecast_test_data[
                "Date"
            ].nunique(),
        },
    ]
)

display(econometric_forecast_checks)
display(forecast_sample_summary)

print(
    "Maximum shock-decomposition error:",
    decomposition_error,
)
print(
    "All econometric forecast checks passed:",
    bool(econometric_forecast_checks["Passed"].all()),
)

,Check,Passed
0,"Development sample contains 4,002 rows",True
1,Development sample covers 87 months,True
2,Development ends in December 2024,True
3,Test sample contains 552 rows,True
4,Test sample covers 12 months,True
5,Test sample covers 46 subclasses,True
6,No duplicate development rows,True
7,No duplicate test rows,True
8,No missing values in aligned data,True
9,Depreciation shocks are non-negative,True


,Sample,Start_Date,End_Date,Observations,Food_Subclasses,Unique_Months
0,Development,2017-10-01,2024-12-01,4002,46,87
1,Locked test,2025-01-01,2025-12-01,552,46,12


Maximum shock-decomposition error: 0.0
All econometric forecast checks passed: True


### Development only ARDL and NARDL lag selection

Separate models are estimated for each food subclass. Candidate food-inflation and exchange-rate lag orders range from one to six, producing 36 candidate specifications per subclass and model type.

All candidates use the same 87-month development sample after the common six-month hold-back. This ensures that BIC comparisons are not affected by different sample sizes.

The symmetric ARDL models include lagged exchange-rate changes. The asymmetric NARDL models replace these changes with separate lagged depreciation and appreciation shocks. Eleven monthly indicators control for seasonality.

No 2025 outcome is used for lag selection or coefficient estimation.

In [32]:
# estimate development-only lag candidates

def evaluate_lag_candidates(
    development_data,
    model_name,
    maximum_price_lag=6,
    maximum_exchange_lag=6,
):
    candidate_records = []
    failure_records = []

    for subclass, subclass_data in development_data.groupby(
        "SubclassDescription",
        sort=True,
    ):
        subclass_data = subclass_data.sort_values("Date")

        class_description = subclass_data[
            "ClassDescription"
        ].iloc[0]

        for price_lag in range(1, maximum_price_lag + 1):
            price_columns = [
                f"Food_Inflation_Pct_Lag{lag}"
                for lag in range(1, price_lag + 1)
            ]

            for exchange_lag in range(
                1,
                maximum_exchange_lag + 1,
            ):
                if model_name == "ARDL":
                    exchange_columns = [
                        (
                            "ExchangeRate_Log_Change_Pct"
                            f"_Lag{lag}"
                        )
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ]
                elif model_name == "NARDL":
                    exchange_columns = [
                        f"Depreciation_Shock_Pct_Lag{lag}"
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ] + [
                        f"Appreciation_Shock_Pct_Lag{lag}"
                        for lag in range(
                            1,
                            exchange_lag + 1,
                        )
                    ]
                else:
                    raise ValueError(
                        f"Unsupported model: {model_name}"
                    )

                predictor_columns = (
                    price_columns
                    + exchange_columns
                    + seasonal_columns
                )

                model_data = subclass_data[
                    ["Food_Inflation_Pct"]
                    + predictor_columns
                ].astype(float)

                design_matrix = sm.add_constant(
                    model_data[predictor_columns],
                    has_constant="add",
                )

                try:
                    fitted_model = sm.OLS(
                        model_data["Food_Inflation_Pct"],
                        design_matrix,
                    ).fit()

                    if not np.isfinite(fitted_model.bic):
                        raise ValueError("Non-finite BIC")

                    candidate_records.append(
                        {
                            "ClassDescription": (
                                class_description
                            ),
                            "SubclassDescription": subclass,
                            "Model": model_name,
                            "Price_Lag": price_lag,
                            "Exchange_Lag": exchange_lag,
                            "BIC": fitted_model.bic,
                            "AIC": fitted_model.aic,
                            "Observations": int(
                                fitted_model.nobs
                            ),
                            "Parameters": int(
                                len(fitted_model.params)
                            ),
                        }
                    )

                except Exception as error:
                    failure_records.append(
                        {
                            "SubclassDescription": subclass,
                            "Model": model_name,
                            "Price_Lag": price_lag,
                            "Exchange_Lag": exchange_lag,
                            "Error": str(error),
                        }
                    )

    return (
        pd.DataFrame(candidate_records),
        pd.DataFrame(failure_records),
    )

In [33]:
# select the minimum-BIC model for each subclass
ardl_lag_candidates, ardl_lag_failures = (
    evaluate_lag_candidates(
        forecast_development_data,
        model_name="ARDL",
    )
)

nardl_lag_candidates, nardl_lag_failures = (
    evaluate_lag_candidates(
        forecast_development_data,
        model_name="NARDL",
    )
)

econometric_lag_candidates = pd.concat(
    [
        ardl_lag_candidates,
        nardl_lag_candidates,
    ],
    ignore_index=True,
)

econometric_lag_failures = pd.concat(
    [
        ardl_lag_failures,
        nardl_lag_failures,
    ],
    ignore_index=True,
)

successful_candidate_counts = (
    econometric_lag_candidates.groupby(
        ["SubclassDescription", "Model"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Successful_Candidates"})
)

selected_econometric_lags = (
    econometric_lag_candidates.sort_values(
        [
            "SubclassDescription",
            "Model",
            "BIC",
        ]
    )
    .drop_duplicates(
        subset=["SubclassDescription", "Model"],
        keep="first",
    )
    .merge(
        successful_candidate_counts,
        on=["SubclassDescription", "Model"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["Model", "SubclassDescription"]
    )
    .reset_index(drop=True)
)

candidate_validation = (
    successful_candidate_counts.groupby(
        "Model",
        as_index=False,
    )
    .agg(
        Food_Subclasses=(
            "SubclassDescription",
            "nunique",
        ),
        Minimum_Successful_Candidates=(
            "Successful_Candidates",
            "min",
        ),
        Maximum_Successful_Candidates=(
            "Successful_Candidates",
            "max",
        ),
    )
)

selected_lag_summary = (
    selected_econometric_lags.groupby(
        [
            "Model",
            "Price_Lag",
            "Exchange_Lag",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "Food_Subclasses"})
    .sort_values(
        [
            "Model",
            "Food_Subclasses",
            "Price_Lag",
            "Exchange_Lag",
        ],
        ascending=[True, False, True, True],
    )
    .reset_index(drop=True)
)

print(
    "Successful lag candidates:",
    len(econometric_lag_candidates),
)
print(
    "Expected lag candidates:",
    46 * 36 * 2,
)
print(
    "Failed lag candidates:",
    len(econometric_lag_failures),
)
print(
    "Selected econometric specifications:",
    len(selected_econometric_lags),
)

display(candidate_validation)
display(selected_lag_summary)

print(
    "All subclasses have 36 successful candidates:",
    bool(
        successful_candidate_counts[
            "Successful_Candidates"
        ].eq(36).all()
    ),
)

Successful lag candidates: 3312
Expected lag candidates: 3312
Failed lag candidates: 0
Selected econometric specifications: 92


,Model,Food_Subclasses,Minimum_Successful_Candidates,Maximum_Successful_Candidates
0,ARDL,46,36,36
1,NARDL,46,36,36


,Model,Price_Lag,Exchange_Lag,Food_Subclasses
0,ARDL,1,1,33
1,ARDL,1,2,3
2,ARDL,2,1,3
3,ARDL,3,1,3
4,ARDL,3,2,2
5,ARDL,2,3,1
6,ARDL,5,1,1
7,NARDL,1,1,34
8,NARDL,2,1,4
9,NARDL,3,1,4


All subclasses have 36 successful candidates: True
